# Week 5 · Demo — The Eval Problem
### Why the testing you learned in Week 3 can't tell you if an LLM's answer is any *good*

This is the **mental-shift** demo of Week 5. By the end of this notebook you'll have *felt*, in running code, why `assert answer == expected` is useless for LLM outputs — and what you do instead.

**How to use this notebook:** run the cells top to bottom. Re-run them, change the question, change the temperature — that's the point of a notebook. Each section builds on the last.

**No API key? No problem.** This notebook runs fully in **fake mode** using three realistic canned responses, so every lesson lands deterministically. Set a key and flip one switch to see it happen against the *real* `gpt-4o-mini`.

---
### The scenario
A company has an HR assistant built on your `/ask` endpoint. An employee asks:

> **"What is the company's policy on remote work?"**

The *actual* policy: employees may work remotely **up to 3 days per week, with manager approval.** Hold onto that — the "3 days" detail is the thing that will separate a good answer from a merely plausible one.

## 0 · Setup

We define an `ask_three(question)` helper that returns **three** answers to the same question.
- **Fake mode** (default): three hand-written responses that are all *plausible* — two fully correct, one subtly incomplete. Deterministic, no key needed.
- **Real mode**: three live calls to `gpt-4o-mini` at `temperature=0.7` (the randomness is deliberate — we *want* to see it vary).

Flip `USE_FAKE = False` and set `OPENAI_API_KEY` to go live.

In [1]:
import os

USE_FAKE = True   # <-- set False (and export OPENAI_API_KEY) to call the real model

QUESTION = "What is the company's policy on remote work?"

# The three canned responses used in fake mode.
# R1: concise + correct   R2: verbose + correct   R3: plausible but MISSING the "3 days" detail
CANNED = [
    "Employees may work remotely up to 3 days per week, with manager approval.",
    "Our remote-work policy lets team members work from home for up to three days "
    "each week. You'll want to coordinate with your manager for coverage, but the "
    "standard allowance is three remote days weekly.",
    "Employees are allowed to work remotely with manager approval, as described in "
    "the employee handbook.",
]

def _call_openai(question, temperature=0.7):
    from openai import OpenAI
    client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])
    resp = client.chat.completions.create(
        model="gpt-4o-mini",
        temperature=temperature,
        messages=[
            {"role": "system", "content":
             "You are an HR assistant. Policy: employees may work remotely up to "
             "3 days per week with manager approval. Answer the user's question."},
            {"role": "user", "content": question},
        ],
    )
    return resp.choices[0].message.content.strip()

def ask_three(question):
    if USE_FAKE:
        return list(CANNED)
    return [_call_openai(question) for _ in range(3)]

print("mode:", "FAKE (canned)" if USE_FAKE else "REAL (gpt-4o-mini)")

mode: FAKE (canned)


## 1 · Ask the same question three times

Run the cell. Same question, three answers. Notice they're **all different wordings** — and in real mode they'll differ *every time you re-run*. That variation isn't a bug; it's the nature of the thing. Re-run it a few times.

In [2]:
answers = ask_three(QUESTION)
for i, a in enumerate(answers, 1):
    print(f"--- Response {i} ---\n{a}\n")

--- Response 1 ---
Employees may work remotely up to 3 days per week, with manager approval.

--- Response 2 ---
Our remote-work policy lets team members work from home for up to three days each week. You'll want to coordinate with your manager for coverage, but the standard allowance is three remote days weekly.

--- Response 3 ---
Employees are allowed to work remotely with manager approval, as described in the employee handbook.



Read them. **Response 1** is concise and correct. **Response 2** is verbose and correct. **Response 3**… sounds fine, but look closely — it never says **3 days**. It's *plausible* and *incomplete*. A human skimming would nod at all three; only a careful reader catches that R3 dropped the one detail that matters.

Now — how would a *test* tell them apart?

## 2 · The assertion trap

In Week 3 you tested code with `assert`. The instinct is to do the same here: decide the "expected" answer and assert equality.

In [3]:
expected = "Employees may work remotely 3 days a week."

for i, a in enumerate(answers, 1):
    try:
        assert a == expected
        print(f"Response {i}: PASS")
    except AssertionError:
        print(f"Response {i}: FAIL  (assert answer == expected)")

Response 1: FAIL  (assert answer == expected)
Response 2: FAIL  (assert answer == expected)
Response 3: FAIL  (assert answer == expected)


**Every response fails — including the two that are perfectly correct.**

The assertion isn't checking *correctness*, it's checking *character-for-character identity*. R1 and R2 are right and useful; they fail because they're worded differently from our one frozen string. This is the whole problem in three lines: an exact-match test on a system that *never* produces the exact same string twice.

"But surely," someone says, "if I normalize case and whitespace it'll work?" Let's try.

In [4]:
def normalize(s):
    return " ".join(s.lower().split())

for i, a in enumerate(answers, 1):
    print(f"Response {i}: {'PASS' if normalize(a) == normalize(expected) else 'FAIL'}")

Response 1: FAIL
Response 2: FAIL
Response 3: FAIL


Still all fail. Normalization removes trivial differences, but the answers differ in *substance and phrasing*, not just case. You cannot normalize your way out of this — the outputs are legitimately varied. Exact-match, in any form, is the wrong tool.

## 3 · A keyword check — a crude *property*

Here's the first real idea. Instead of asking *"is it this exact string?"*, ask *"does it contain the fact that matters?"* — the **3 days** detail.

In [5]:
def mentions_three_days(a):
    text = a.lower()
    return "3 days" in text or "three days" in text

for i, a in enumerate(answers, 1):
    print(f"Response {i}: {'has the 3-day detail' if mentions_three_days(a) else 'MISSING the 3-day detail'}")

Response 1: has the 3-day detail
Response 2: has the 3-day detail
Response 3: MISSING the 3-day detail


Now something useful happens: **R1 and R2 pass, R3 is flagged.** The keyword check caught exactly the incompleteness a human noticed — the thing `assert` was blind to. We stopped checking *identity* and started checking a *property* ("does it state the 3-day limit?").

But keyword checks are brittle in two directions. They **miss** correct answers that use different words, and they **pass** wrong answers that happen to contain the keyword. Watch: 

In [6]:
sneaky_wrong = "There is no limit — employees can work remotely all 3 days, 4 days, or 5 days, whenever they like."
print("mentions '3 days'? ", mentions_three_days(sneaky_wrong))
print("...but the answer is WRONG — it says there's no limit.")

mentions '3 days'?  True
...but the answer is WRONG — it says there's no limit.


The keyword check says PASS on an answer that *contradicts the policy*, just because the string "3 days" appears in it. So a keyword is a *property*, but a weak one. We need properties that capture **meaning**, not just substrings.

## 4 · Evaluate *properties*, not strings

This is the shift. We stop asking "does it equal X?" and start asking "does it have the right **properties**?" Three properties carry most of the weight — the same three you'll build into tomorrow's judge and rubric:

- **Accurate** — does it state the correct facts? (here: the 3-day limit *and* manager approval, without contradicting them)
- **Grounded** — does it stay within the source policy rather than inventing new rules?
- **Format** — is it reasonably concise? (say, under 60 words for a chat answer)

For this demo we implement them as simple deterministic checks so you can see the mechanism. (Tomorrow, an LLM *judge* generalizes these so you don't hand-write a checker per question.)

In [7]:
def is_accurate(a):
    t = a.lower()
    has_limit = "3 days" in t or "three days" in t
    has_approval = "manager" in t or "approval" in t
    contradicts = any(p in t for p in ["no limit", "unlimited", "4 days", "5 days", "any day"])
    return has_limit and has_approval and not contradicts

def is_grounded(a):
    # grounded = doesn't invent rules outside the known policy (toy check for the demo)
    invented = any(p in a.lower() for p in ["stipend", "reimburse", "equipment budget", "unlimited"])
    return not invented

def follows_format(a, max_words=60):
    return len(a.split()) <= max_words

def evaluate(a):
    return {"accurate": is_accurate(a), "grounded": is_grounded(a), "format": follows_format(a)}

# score the three real answers + the sneaky-wrong one
labels = ["Response 1", "Response 2", "Response 3", "Sneaky-wrong"]
samples = answers + [sneaky_wrong]

print(f"{'':13} {'accurate':>9} {'grounded':>9} {'format':>7}")
for label, a in zip(labels, samples):
    p = evaluate(a)
    print(f"{label:13} {str(p['accurate']):>9} {str(p['grounded']):>9} {str(p['format']):>7}")

               accurate  grounded  format
Response 1         True      True    True
Response 2         True      True    True
Response 3        False      True    True
Sneaky-wrong      False      True    True


Look at what the property table does that `assert` could not:

- **R1 and R2** — accurate, grounded, well-formatted. ✅ Correctly recognised as *good*, despite being different strings.
- **R3** — fails **accuracy** (no 3-day limit). Correctly flagged as *incomplete*.
- **Sneaky-wrong** — fails **accuracy** (it contradicts the policy), even though it contains "3 days". The property check saw the *meaning* the keyword check missed.

That's the mental shift, made concrete: **valid answers vary in wording but share properties; you evaluate the properties.**

## 5 · …but this doesn't scale — which is why the *judge* exists

Notice what we just did: we hand-wrote `is_accurate` for *one* question, baking in "3 days" and "manager". That works for a demo and **falls apart at 20 questions**, let alone 200 — you can't hand-code a checker for every question in your golden set.

Two things fix that, and they're the rest of Week 5:

1. **The golden set** — instead of hard-coding facts in Python, you store, per question, an `ideal_answer` and `notes` about what a good answer must contain. (That's the next demo, and Lab Step 2.)
2. **The LLM-as-judge** — instead of hand-writing `is_accurate`, you *ask a strong model* (`gpt-4o`) to score an answer against the ideal answer on accuracy, groundedness, and format. One judge, any question. (That's tomorrow.)

So this notebook is the *why*. The judge is the *how, at scale*.

## 6 · Your turn — experiment

A notebook is for poking at things. Try these:

1. **Go live.** Set `USE_FAKE = False`, export `OPENAI_API_KEY`, re-run from the top. Watch the three real answers differ — and differ again each re-run.
2. **Crank the temperature.** In `_call_openai`, change `temperature=0.7` to `1.3`. The answers get wilder — and exact-match gets even more hopeless.
3. **Break the properties.** Write your own `sneaky_wrong` answer that passes all three property checks but is actually bad. (This is exactly the game an eval engineer plays against their own judge.)
4. **Your domain.** Swap `QUESTION` and the policy for *your* capstone — a banking FAQ, a medical triage line, a legal clause lookup — and rewrite the property checks. Notice how the *shape* stays the same even though the facts change.

## Fit it into your app

This demo doesn't add code to your app directly — it changes how you *think* about the next two things you build:

- **`data/golden_set.jsonl`** — each entry stores a `question`, an `ideal_answer`, and `notes` (the properties that matter). That's the durable, per-question version of the facts we hard-coded in `is_accurate`.
- **`src/eval/judge.py`** (tomorrow) — an LLM-as-judge that scores your `/ask` answers against those ideal answers on accuracy, groundedness, and format — the generalization of section 4.

**The one-line takeaway to carry into the build:** *stop testing whether the answer is a specific string; start evaluating whether it has the right properties.*